# questão 6 - previsão de demanda
**Produto:** Bussola de Bordo 702
**Modelo:** baseline (média móvel de 3 meses)

In [7]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

### 1 - carregamento e unificação dos dados

In [ ]:
# carregar os datasets
orders = pd.read_csv('../lh_nautical_csv/orders.csv')
order_items = pd.read_csv('../lh_nautical_csv/order_items.csv')
product_variants = pd.read_csv('../lh_nautical_csv/product_variants.csv')
products = pd.read_csv('../lh_nautical_csv/products.csv')

# unificar os datasets
df = pd.merge(order_items, product_variants, left_on='product_variant_id', right_on='id', suffixes=('_item', '_var'))
df = pd.merge(df, products, left_on='product_id', right_on='id', suffixes=('', '_prod'))
# adicionar _order nas colunas de orders para não confundir com as datas do catalogo
df = pd.merge(df, orders, left_on='order_id', right_on='id', suffixes=('', '_order'))

# filtrar o produto e puxando a data de venda CORRETA
df_bussola = df[df['name'] == 'Bússola de Bordo 702'].copy()
df_bussola['created_at'] = pd.to_datetime(df_bussola['created_at_order'])
df_bussola['year_month'] = df_bussola['created_at'].dt.to_period('M').dt.to_timestamp()

# agrupar vendas por mes
monthly_sales = df_bussola.groupby('year_month')['quantity'].sum().reset_index()

# preencher meses sem vendas com 0
all_months = pd.date_range(start='2020-01-01', end='2026-03-01', freq='MS')
monthly_sales = monthly_sales.set_index('year_month').reindex(all_months, fill_value=0).reset_index()
monthly_sales.columns = ['year_month', 'quantity']

# media movel de 3 meses com SHIFT para não usar o mes previsto no calculo (evitando data leakage)
monthly_sales['pred_3m_ma'] = monthly_sales['quantity'].rolling(window=3).mean().shift(1)

### 2 - construção do dataset mensal e modelo baseline

In [ ]:
# agrupar vendas por mes
monthly_sales = df_bussola.groupby('year_month')['quantity'].sum().reset_index()

# preencher meses sem vendas com 0 (começo 2020 - março 2026)
all_months = pd.date_range(start='2020-01-01', end='2026-03-01', freq='MS')
monthly_sales = monthly_sales.set_index('year_month').reindex(all_months, fill_value=0).reset_index()
monthly_sales.columns = ['year_month', 'quantity']

# media movel de 3 meses
# shift(1) para que a previsão do mes atual use apenas os 3 meses anteriores (evitar data leakage)
monthly_sales['pred_3m_ma'] = monthly_sales['quantity'].rolling(window=3).mean().shift(1)

# visualizar os ultimos meses antes do teste
monthly_sales.tail(10)

,year_month,quantity,pred_3m_ma
65,2025-06-01,17,46.333333
66,2025-07-01,19,26.333333
67,2025-08-01,23,20.000000
68,2025-09-01,31,19.666667
69,2025-10-01,34,24.333333
70,2025-11-01,60,29.333333
71,2025-12-01,22,41.666667
72,2026-01-01,79,38.666667
73,2026-02-01,68,53.666667
74,2026-03-01,60,56.333333


### 3 - período de teste e 4 - avaliação (MAE)

In [ ]:
# filtrar estritamente o primeiro trimestre - 2026
test_period = monthly_sales[(monthly_sales['year_month'] >= '2026-01-01') & (monthly_sales['year_month'] <= '2026-03-01')]

y_true = test_period['quantity']
y_pred = test_period['pred_3m_ma']

# avaliar o erro e totais
mae = mean_absolute_error(y_true, y_pred)
total_previsto = y_pred.sum()
total_real = y_true.sum()

df_display = test_period[['year_month', 'quantity', 'pred_3m_ma']].copy()
df_display['year_month'] = df_display['year_month'].dt.strftime('%m/%Y')
df_display.rename(columns={
    'year_month': 'Mês', 
    'quantity': 'Vendas Reais', 
    'pred_3m_ma': 'Previsão (MM)'
}, inplace=True)
df_display['Previsão (MM)'] = df_display['Previsão (MM)'].round(1)

print("=====================================================")
print(" ⚓  RELATÓRIO EXECUTIVO LH NAUTICAL  ⚓")
print(" 🧭  Produto: Bússola de Bordo 702  🧭")
print(" 📅  Período: Q1 2026")
print("=====================================================")
print("\n📊 COMPARATIVO MENSAL:")
print(df_display.to_string(index=False))
print("\n-----------------------------------------------------")
print("📉 MÉTRICAS DO MODELO (BASELINE):")
print(f"   • MAE (Erro Médio Absoluto): {mae:.2f} unidades")
print(f"   • Previsão Bruta do Trimestre: {total_previsto:.2f}")
print(f"   • Vendas Reais do Trimestre:   {total_real:.2f}")
print(f"   • Sugestão de Compra (Arredondado): {round(total_previsto)} unidades")
print("=====================================================")

 ⚓  RELATÓRIO EXECUTIVO LH NAUTICAL  ⚓
 🧭  Produto: Bússola de Bordo 702  🧭
 📅  Período: Q1 2026

📊 COMPARATIVO MENSAL:
    Mês  Vendas Reais  Previsão (MM)
01/2026            79           38.7
02/2026            68           53.7
03/2026            60           56.3

-----------------------------------------------------
📉 MÉTRICAS DO MODELO (BASELINE):
   • MAE (Erro Médio Absoluto): 19.44 unidades
   • Previsão Bruta do Trimestre: 148.67
   • Vendas Reais do Trimestre:   207.00
   • Sugestão de Compra (Arredondado): 149 unidades
